### GOLD TESTING - FACT TRANSACTION ITEMS (SCD1)

#### Purpose
- Validate `coffee.gold.fact_transaction_items` against `coffee.silver.transaction_items`
- Ensure rollup logic and measures are correct
- Validate key uniqueness for both:
  - `transaction_item_sk` (string surrogate key)
  - `transaction_item_key` (bigint surrogate key)
- Validate referential integrity:
  - transaction_id must exist in `fact_transactions`
  - item_id must exist in current `dim_menu_items`

#### Tests Covered
1. Silver vs Gold row count reconciliation
2. Null checks on keys (`transaction_item_sk`, `transaction_item_key`)
3. Duplicate check on `transaction_item_sk`
4. Uniqueness check on bigint key (`transaction_item_key`)
5. Amount reconciliation (subtotal)
6. Referential integrity:
   - transaction_id → fact_transactions
   - item_id → dim_menu_items


In [0]:
-- TEST 1: SILVER vs GOLD ROW COUNT RECONCILIATION
SELECT
  'fact_transaction_items_count_recon' AS test_name,
  (SELECT COUNT(*) FROM coffee.silver.transaction_items) AS silver_count,
  (SELECT COUNT(*) FROM coffee.gold.fact_transaction_items) AS gold_count;

In [0]:
-- TEST 2: NULL CHECK ON STRING SURROGATE KEY
SELECT
  'fact_transaction_items_null_transaction_item_sk' AS test_name,
  COUNT(*) AS null_key_count
FROM coffee.gold.fact_transaction_items
WHERE transaction_item_sk IS NULL;

In [0]:
-- TEST 3: NULL CHECK ON BIGINT KEY
SELECT
  'fact_transaction_items_null_transaction_item_key' AS test_name,
  COUNT(*) AS null_key_count
FROM coffee.gold.fact_transaction_items
WHERE transaction_item_key IS NULL;

In [0]:
-- TEST 4: DUPLICATE CHECK ON STRING KEY (transaction_item_sk)
SELECT
  'fact_transaction_items_duplicate_transaction_item_sk' AS test_name,
  COUNT(*) AS duplicate_key_count
FROM (
  SELECT transaction_item_sk
  FROM coffee.gold.fact_transaction_items
  GROUP BY transaction_item_sk
  HAVING COUNT(*) > 1
);

In [0]:
-- TEST 5: UNIQUENESS CHECK ON BIGINT KEY (transaction_item_key)
SELECT
  'fact_transaction_items_duplicate_transaction_item_key' AS test_name,
  COUNT(*) AS duplicate_key_count
FROM (
  SELECT transaction_item_key
  FROM coffee.gold.fact_transaction_items
  GROUP BY transaction_item_key
  HAVING COUNT(*) > 1
);

In [0]:
-- TEST 6: AMOUNT RECONCILIATION (SUBTOTAL)
SELECT
  'transaction_items_subtotal' AS metric,
  (SELECT ROUND(SUM(subtotal), 2) FROM coffee.silver.transaction_items) AS silver_sum,
  (SELECT ROUND(SUM(subtotal), 2) FROM coffee.gold.fact_transaction_items) AS gold_sum;

In [0]:
-- TEST 7: REFERENTIAL INTEGRITY - FACT ITEMS -> FACT TRANSACTIONS
SELECT
  'RI_fact_transaction_items_transaction_id' AS test_name,
  COUNT(*) AS missing_fk_count
FROM coffee.gold.fact_transaction_items fi
LEFT JOIN coffee.gold.fact_transactions ft
  ON fi.transaction_id = ft.transaction_id
WHERE ft.transaction_id IS NULL;

In [0]:
-- TEST 8: REFERENTIAL INTEGRITY - FACT ITEMS -> DIM MENU ITEMS
SELECT
  'RI_fact_transaction_items_item_id' AS test_name,
  COUNT(*) AS missing_fk_count
FROM coffee.gold.fact_transaction_items fi
LEFT JOIN coffee.gold.dim_menu_items d
  ON fi.item_id = d.item_id AND d.__END_AT IS NULL
WHERE d.item_id IS NULL;